In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [18]:
# 将建筑和楼层进行合并，形成新的编码


# 1. 读取训练数据（路径请按需修改）
train_df = pd.read_csv('../../UJIIndoorLoc/trainingData.csv')
valid_df = pd.read_csv('../../UJIIndoorLoc/validationData.csv')
train_df_noisy1 = pd.read_csv('../../data/train_noisy1.csv')

# train_df = pd.concat([train_df, train_df_noisy1], ignore_index=True)
# total_df = pd.concat([train_df, valid_df], ignore_index=True)

# 2. 创建联合标签列（如 "2_3" 表示 BUILDINGID=2 且 FLOOR=3）
train_df['location_label'] = train_df['BUILDINGID'].astype(str) + '_' + train_df['FLOOR'].astype(str)
valid_df['location_label'] = valid_df['BUILDINGID'].astype(str) + '_' + valid_df['FLOOR'].astype(str)
# total_df['location_label'] = total_df['BUILDINGID'].astype(str) + '_' + total_df['FLOOR'].astype(str)

# 3. 将联合标签进行整数编码
label_encoder = LabelEncoder()
train_df['location_label_encoded'] = label_encoder.fit_transform(train_df['location_label'])
valid_df['location_label_encoded'] = label_encoder.transform(valid_df['location_label'])  # 改这里
# total_df['location_label_encoded'] = label_encoder.fit_transform(valid_df['location_label'])
total_df = pd.concat([train_df, valid_df])
# 4. 保存为 CSV 文件
# train_df.to_csv('./data/processed_train.csv', index=False)
# valid_df.to_csv('./data/processed_valid.csv', index=False)

# print("✅ 文件已成功保存为 'processed_train.csv'")
# print("✅ 文件已成功保存为 'processed_validat.csv'")

In [19]:
# 取特征和标签
# 训练集
training_data = train_df[train_df.columns[:520]].to_numpy()
training_floors = train_df['location_label_encoded'].to_numpy() # FLOOR LABELS
training_longitude = train_df['LONGITUDE'].to_numpy() # LONGITUDE LABELS
training_latitude = train_df['LATITUDE'].to_numpy() # LATITUDE LABELS
# 验证集
valid_data = valid_df[valid_df.columns[:520]].to_numpy()
valid_floors = valid_df['location_label_encoded'].to_numpy() # FLOOR LABELS
valid_longitude = valid_df['LONGITUDE'].to_numpy() # LONGITUDE LABELS
valid_latitude = valid_df['LATITUDE'].to_numpy() # LATITUDE LABELS
# 数据总和
total_data = total_df[total_df.columns[:520]].to_numpy()
total_floors = total_df['location_label_encoded'].to_numpy() # FLOOR LABELS
total_longitude = total_df['LONGITUDE'].to_numpy() # LONGITUDE LABELS
total_latitude = total_df['LATITUDE'].to_numpy() # LATITUDE LABELS



In [20]:
# 数据归一化处理
import sys
sys.path.append('..')

from utill.data_standar import normalize_rssi, normalize_coords, normalize_test_or_valid_data
# 数据标准化,从总数居获取最大值最小值
X_totalCo_cnn, X_min, X_max = normalize_rssi(total_data)
print(X_min, X_max)
# 训练集特征标准化
training_data = normalize_test_or_valid_data(X_min, X_max, training_data)
# 验证集特征标准化
valid_data = normalize_test_or_valid_data(X_min, X_max, valid_data)
# 全集
total_data = normalize_test_or_valid_data(X_min, X_max, total_data)

-104 100


# PCA方式降维

In [30]:
# 使用pca进行降维
from sklearn.decomposition import PCA
pca = PCA(n_components=30)
X_trainF_reduced = pca.fit_transform(training_data)
X_testF_reduced = pca.transform(valid_data) 

In [31]:
from KNN import KNN_clf

In [32]:
# Testing Floor Classification (pca n_components=10)
for n in range(1,6):
    knn = KNN_clf(n_neighbours=n)
    knn.fit(X_trainF_reduced, training_floors)
    preds = knn.predict(X_testF_reduced)
    print(f'Model Accuracy (KNN={n}): {knn.accuracy_metric(preds, valid_floors) *100}%' )

Model Accuracy (KNN=1): 75.42754275427542%
Model Accuracy (KNN=2): 75.42754275427542%
Model Accuracy (KNN=3): 74.88748874887489%
Model Accuracy (KNN=4): 75.33753375337534%
Model Accuracy (KNN=5): 75.06750675067508%


In [24]:
from KNN import KNN_reg

In [33]:
# Longitude Regression (pca n_components=10)
for n in range(1,6):
    knn = KNN_reg(n_neighbours=n)
    knn.fit(X_trainF_reduced, training_longitude)
    preds = knn.predict(X_testF_reduced)
    print(f'(KNN={n})')
    print(f'Test MSE: {knn.MSE_metric(preds, valid_longitude)}')
    print(f'Test R^2: {knn.r2_metric(preds, valid_longitude)}\n')


(KNN=1)
Test MSE: 266.99120556058557
Test R^2: 0.9815068181323425

(KNN=2)
Test MSE: 228.31277044956587
Test R^2: 0.98418588515765

(KNN=3)
Test MSE: 201.7818146155664
Test R^2: 0.9860235553922586

(KNN=4)
Test MSE: 189.57049188466294
Test R^2: 0.9868693743084027

(KNN=5)
Test MSE: 207.435518822048
Test R^2: 0.9856319508077672



In [26]:
#  Latitude Regression (pca n_components=10)
for n in range(1,6):
    knn = KNN_reg(n_neighbours=n)
    knn.fit(X_trainF_reduced, training_latitude)
    preds = knn.predict(X_testF_reduced)
    print(f'(KNN={n})')
    print(f'Test MSE: {knn.MSE_metric(preds, valid_latitude)}')

(KNN=1)
Test MSE: 290.7915672936205
(KNN=2)
Test MSE: 238.83085585694744
(KNN=3)
Test MSE: 219.84355804528315
(KNN=4)
Test MSE: 219.7170320431578
(KNN=5)
Test MSE: 238.6100464709594
